In [1]:
import numpy as np

#eps' ~= (x - x') / (1 - 1/D),   D = (1+omega)*f'(x) - omega

def solve(f, fprime, x0, omega=0.0, accuracy=1e-6, max_iter=10000):
#Solve x = f(x) by overrelaxation, using the error estimate from
#part (a) as the stopping test.
#Returns: solution, number of iterations, history of x values.
    x1 = x0
    x2 = (1 + omega) * f(x1) - omega * x1
    history = [x1, x2]
    iterations = 1

    def D(x):
        return (1 + omega) * fprime(x) - omega

    def error_estimate(x_old, x_new):
        return (x_old - x_new) / (1 - 1 / D(x_old))

    while abs(error_estimate(x1, x2)) > accuracy and iterations < max_iter:
        x1, x2 = x2, (1 + omega) * f(x2) - omega * x2
        history.append(x2)
        iterations += 1

    return x2, iterations, history

#Parts (b) and (c):
c = 2.0
f_c2       = lambda x: 1 - np.exp(-c * x)
fprime_c2  = lambda x: c * np.exp(-c * x)

print("x = 1 - exp(-2x)")
x_sol, n_iter, _ = solve(f_c2, fprime_c2, x0=0.5, omega=0.0, accuracy=1e-6)
print(f"ordinary relaxation (omega=0):   x = {x_sol:.8f}   iterations = {n_iter}")

x_sol, n_iter, _ = solve(f_c2, fprime_c2, x0=0.5, omega=0.5, accuracy=1e-6)
print(f"overrelaxation (omega=0.5):      x = {x_sol:.8f}   iterations = {n_iter}")

print("\nScanning omega:")
best_omega, best_n = None, np.inf
for omega in np.arange(0.0, 1.0, 0.1):
    x_sol, n_iter, _ = solve(f_c2, fprime_c2, x0=0.5, omega=omega, accuracy=1e-6)
    print(f"  omega={omega:4.1f}  ->  iterations={n_iter:3d}   x={x_sol:.12f}")
    if n_iter < best_n:
        best_omega, best_n = omega, n_iter
print(f"Near-optimal omega found: {best_omega:.1f}  ({best_n} iterations)")


# Part (d): demonstrating that omega < 0 helps when f'(x*) < 0

print("\nDemonstration for part (d): f(x) = 1.5 - 0.5x, x* = 1")
f_lin      = lambda x: 1.5 - 0.5 * x
fprime_lin = lambda x: -0.5 * np.ones_like(x) if hasattr(x, "__len__") else -0.5

x_sol, n_iter, hist = solve(f_lin, fprime_lin, x0=0.0, omega=0.0, accuracy=1e-6)
print(f"ordinary relaxation (omega=0):      x = {x_sol:.6f}  iterations = {n_iter}")
print(f"iterate history: {['%.4f' % v for v in hist[:8]]} ...  (note the sign-flipping oscillation)")

omega_opt = -0.5 / 1.5
x_sol, n_iter, hist = solve(f_lin, fprime_lin, x0=0.0, omega=omega_opt, accuracy=1e-6)
print(f"optimal negative omega ({omega_opt:.4f}):   x = {x_sol:.6f}  iterations = {n_iter}")

print("\nScanning omega (including negative values) for the linear example:")
for omega in [-0.6, -0.5, -1/3, -0.2, 0.0, 0.2, 0.5]:
    x_sol, n_iter, _ = solve(f_lin, fprime_lin, x0=0.0, omega=omega, accuracy=1e-6)
    print(f"  omega={omega:7.4f}  ->  iterations={n_iter:3d}   x={x_sol:.6f}")


x = 1 - exp(-2x)
ordinary relaxation (omega=0):   x = 0.79681138   iterations = 15
overrelaxation (omega=0.5):      x = 0.79681183   iterations = 7

Scanning omega:
  omega= 0.0  ->  iterations= 15   x=0.796811378194
  omega= 0.1  ->  iterations= 13   x=0.796811489795
  omega= 0.2  ->  iterations= 11   x=0.796811368470
  omega= 0.3  ->  iterations= 10   x=0.796811819805
  omega= 0.4  ->  iterations=  8   x=0.796811442845
  omega= 0.5  ->  iterations=  7   x=0.796811829356
  omega= 0.6  ->  iterations=  6   x=0.796812063206
  omega= 0.7  ->  iterations=  4   x=0.796811929558
  omega= 0.8  ->  iterations=  5   x=0.796811675413
  omega= 0.9  ->  iterations=  7   x=0.796811983650
Near-optimal omega found: 0.7  (4 iterations)

Demonstration for part (d): f(x) = 1.5 - 0.5x, x* = 1
ordinary relaxation (omega=0):      x = 0.999999  iterations = 20
iterate history: ['0.0000', '1.5000', '0.7500', '1.1250', '0.9375', '1.0312', '0.9844', '1.0078'] ...  (note the sign-flipping oscillation)
optimal 